# QCoDeS Example with QuTech S4c Source Module

This notebook explains how the QuTech S4c voltage/current source module works and shows the main features of its QCoDeS driver.

## QuTech S4c Current/Voltage Source

The S4c is a versatile voltage and current source designed for use in the IVVI rack system. It provides precise control over voltage and current output with multiple range settings.

**Key Features:**
- Source modes: Voltage (V), Current (I), Voltage with series resistance (V+R)
- Multiple ranges: 1n, 10n, 100n, 1µ, 10µ, 100µ, 1m, 10m, 20m
- Output modes: Symmetric or single-ended
- Optional x0.01 control input for dV/dI measurements
- Adjustable output resistance in V+R mode
- Compatible with IVVI rack system

**Documentation:** https://qtwork.tudelft.nl/~schouten/ivvi/doc-mod/docs4c.htm

## Virtual Driver

This is a **virtual driver** - it does not communicate with the physical instrument. It is the user's responsibility to manually set the physical instrument to match the driver settings. The driver helps track settings and calculate the total output value.

## Quick Start: Voltage Source Example

### Import Required Libraries

In [47]:
from qcodes_contrib_drivers.drivers.QuTech.S4c import S4c, SourceParameter
import numpy as np

### Create an S4c Instance

In [48]:
source = S4c("s4c_source")

### Setting Source Mode

The S4c can operate in three different modes:
- **"V"**: Voltage source mode
- **"I"**: Current source mode
- **"V+R"**: Voltage source with series resistance

In [49]:
# Set to voltage source mode
source.source_mode("V")

### Setting Range

The S4c supports multiple output ranges. Choose the appropriate range for your measurement:
- **"1n", "10n", "100n"**: Nano range (1nV/A to 100nV/A)
- **"1u", "10u", "100u"**: Micro range (1µV/A to 100µV/A)
- **"1m", "10m", "20m"**: Milli range (1mV/A to 20mV/A)

In [50]:
# Set range to 10mV (voltage mode)
source.range("10m")

### Setting Output Mode

The S4c can operate in two output modes:
- **"symm"**: Symmetric output (default)
- **"single"**: Single-ended output

In [51]:
# Set to symmetric output mode
source.output_mode("symm")

### Using SourceParameter for Voltage/Current Control

The `SourceParameter` class automatically scales the voltage or current values according to the S4c's range setting. This allows you to work with the actual output values rather than the normalized control values.

In [52]:
from qcodes.instrument_drivers.mock_instruments import DummyInstrument
# Create a mock DAC (digital-to-analog converter) that will control the S4c
# In a real setup, this would be an actual DAC like QDevil QDAC-II
dac = DummyInstrument(name="dac", gates=["voltage"])

In [53]:
# Create SourceParameter that links the DAC to S4c
voltage_source = SourceParameter(
    source_param=dac.voltage,  # Use the DAC voltage parameter
    source_instrument=source,
    name="voltage_output"
)

In [54]:
# Simulate a voltage sweep
print("Voltage sweep using S4c source:")
print("\nDesired Output (mV) | DAC Control (V) | Actual Output (mV)")
print("-" * 65)

for v_desired in np.linspace(-5e-3, 5e-3, 11):  # Output voltages in mV
    # Set the desired voltage (will automatically calculate DAC control value)
    voltage_source(v_desired)
    
    # Get the values (DAC control and actual output)
    dac_control, actual_output = voltage_source.get()
    
    print(f"{v_desired*1e3:19.2f} | {dac_control:15.2f} | {actual_output*1e3:18.2f}")

print("\nSweep complete!")

Voltage sweep using S4c source:

Desired Output (mV) | DAC Control (V) | Actual Output (mV)
-----------------------------------------------------------------
              -5.00 |           -0.50 |              -5.00
              -4.00 |           -0.40 |              -4.00
              -3.00 |           -0.30 |              -3.00
              -2.00 |           -0.20 |              -2.00
              -1.00 |           -0.10 |              -1.00
               0.00 |            0.00 |               0.00
               1.00 |            0.10 |               1.00
               2.00 |            0.20 |               2.00
               3.00 |            0.30 |               3.00
               4.00 |            0.40 |               4.00
               5.00 |            0.50 |               5.00

Sweep complete!


## Current Source Mode Example

The S4c can also be used as a current source. Let's demonstrate current sourcing:

In [55]:
# Create a new S4c instance for current sourcing
current_source_module = S4c("s4c_current")

# Set to current source mode
current_source_module.source_mode("I")

# Set range to 1µA
current_source_module.range("1u")

# Create a mock DAC for current control
dac_current = DummyInstrument(name="dac_current", gates=["voltage"])

# Create SourceParameter for current
current_source = SourceParameter(
    source_param=dac_current.voltage,
    source_instrument=current_source_module,
    name="current_output"
)

In [56]:
# Simulate a current sweep
print("Current sweep using S4c source:")
print("\nDesired Output (nA) | DAC Control (V) | Actual Output (nA)")
print("-" * 65)

for i_desired in np.linspace(-5e-7, 5e-7, 11):  # Output currents in nA
    # Convert nA to A
    i_desired_amps = i_desired
    
    # Set the desired current
    current_source(i_desired)
    
    # Get the values
    dac_control, actual_output = current_source.get()
    
    print(f"{i_desired*1e9:19.1f} | {dac_control:15.6f} | {actual_output*1e9:18.1f}")

print("\nSweep complete!")

Current sweep using S4c source:

Desired Output (nA) | DAC Control (V) | Actual Output (nA)
-----------------------------------------------------------------
             -500.0 |       -0.500000 |             -500.0
             -400.0 |       -0.400000 |             -400.0
             -300.0 |       -0.300000 |             -300.0
             -200.0 |       -0.200000 |             -200.0
             -100.0 |       -0.100000 |             -100.0
                0.0 |        0.000000 |                0.0
              100.0 |        0.100000 |              100.0
              200.0 |        0.200000 |              200.0
              300.0 |        0.300000 |              300.0
              400.0 |        0.400000 |              400.0
              500.0 |        0.500000 |              500.0

Sweep complete!


## Advanced Features

### Voltage with Series Resistance (V+R) Mode

In V+R mode, the S4c acts as a voltage source with a series resistance. This is useful for creating controlled current sources with voltage compliance.

In [57]:
# Create S4c instance for V+R mode
vr_source = S4c("s4c_vr")

# Set to V+R mode
vr_source.source_mode("V+R")

# Set output resistance
# Options: "R/1000", "R/100", "R/10", "10R", "100R", "1000R"
vr_source.R_out("R/10")

# Set voltage range
vr_source.range("10m")

print("V+R mode configured:")
print(f"  Source mode: {vr_source.source_mode()}")
print(f"  Output resistance: {vr_source.R_out()}")
print(f"  Range: {vr_source.range()}")

V+R mode configured:
  Source mode: V+R
  Output resistance: R/10
  Range: 10m


### Fine Control with x0.01 Input

The S4c has an optional x0.01 control input that allows for fine adjustment of the output:

In [58]:
# Enable the x0.01 control input
source.x001_jumper(True)

print(f"x0.01 control jumper installed: {source.x001_jumper()}")
print("This allows for 100x finer control resolution.")

x0.01 control jumper installed: True
This allows for 100x finer control resolution.


### Checking All Settings

You can view all current settings of the S4c module:

In [59]:
source.print_readable_snapshot()

s4c_source:
	parameter     value
--------------------------------------------------------------------------------
IDN            :	None 
R_out          :	R/10 
output_mode    :	symm 
range          :	10m 
source_mode    :	V 
voltage_output :	(np.float64(0.5), np.float64(0.005)) (V)
x001_jumper    :	True 


## Cleanup

In [60]:
# Clean up mock instruments
dac.close()
source.close()

dac_current.close()

# Close all S4c instances
current_source_module.close()
vr_source.close()